# Transaction-Time-Safe Behavioural Features

Bitcoin Transaction Forensics -- constructs a 14-feature transaction-time-safe behavioural
representation, designed to replace the temporal-availability concerns identified in the
inherited-feature audit (`03_Feature_Temporal_Audit.ipynb`) without reusing any full-history
wallet summary.

Uses **only** `txs_features.csv`, `AddrTx_edgelist.csv`, `TxAddr_edgelist.csv`. Does **not**
load `wallets_features.csv`, does **not** use labels, does **not** train any model.

## Design decisions (approved)

1. **FIN2 (output value concentration / HHI) was dropped, not approximated.** `txs_features.csv`
   only provides five summary statistics of the output distribution (min/max/mean/median/total),
   not individual output values, and `TxAddr_edgelist.csv` carries no per-address value column.
   A true HHI needs every individual output value; no available file provides that. Rather than
   ship a proxy metric that isn't actually what it claims to be, FIN2 is documented as infeasible
   with current data and excluded. Final set: **14 features**.

2. **`wallets_features.csv` is not used anywhere in this notebook.** The entire prior audit
   established that its per-address fields (`lifetime_in_blocks`, `num_txs_as_sender`,
   `btc_sent_total`, etc.) are static full-history snapshots, not point-in-time state. Using it
   here would silently reintroduce the exact problem this notebook exists to fix.

3. **Why the temporal features are transaction-time-safe:** every TMP/NET feature is built from
   `AddrTx_edgelist.csv` / `TxAddr_edgelist.csv` restricted by each transaction's own `Time step`
   -- never from a static summary table. Each feature's exact information boundary is stated
   explicitly in its own section below, with an assertion enforcing it.

4. **Why TMP6 uses strictly `< t` while TMP1/TMP2/TMP4/TMP5/TMP7 use `<= t`:** same-time-step
   chronological order cannot be resolved in this dataset (established in earlier project work --
   the transaction graph itself only contains same-time-step edges). For age and count features,
   the documented fallback boundary of "available within or before the transaction's own time
   step" (`<= t`) is used, with this limitation stated explicitly rather than assumed. TMP6
   ("time since last activity") is the one feature designed to avoid this ambiguity entirely --
   it is built from `shift(1)` on a sorted, deduplicated per-address time-step series, which is
   mechanically restricted to a *strictly prior* distinct time step and can never reference the
   current or a future one.

5. **Why NET1/NET2 use the address graph, not `txs_edgelist.csv`:** the transaction-to-transaction
   edge list was independently confirmed (in earlier project work) to contain **only** same-time-
   step edges -- 0 of 234,355 edges cross time steps. It cannot express genuine historical
   structure. The address graph (`AddrTx_edgelist.csv` / `TxAddr_edgelist.csv`) can, because the
   same address can legitimately appear at different time steps, giving real cross-time links.

6. **Why MIN is used to collapse multiple addresses into one transaction-level value:** this is a
   **deliberate forensic aggregation choice, not a mathematical requirement.** It is chosen for
   consistency with the minimum-based semantics of the inherited features being replaced
   (`Local_feature_53` = minimum lifetime, `Local_feature_41` = minimum incoming transactions),
   and because the minimum captures the least-experienced / youngest / most-dormant address in
   the transaction -- the conservative choice for forensic screening. **This is not claimed to be
   objectively optimal** -- a mean- or max-based variant is an equally valid alternative for a
   different research question.

## Important correction applied in this implementation

For TMP1/TMP2 (address age), a full-history `groupby("address")["Time step"].min()` is
mathematically provable to equal an explicit `<= t` restricted computation, **for the minimum
operation only** (proof: if an address is active at time `t`, `t` is a member of its activity set,
so the true minimum of that set can never exceed `t` -- restricting to `<= t` cannot change the
answer). **This proof does NOT generalize to COUNT (TMP4/TMP5/TMP7) or to TMP6** -- those
genuinely differ between "full history" and "restricted to t", and are implemented with explicit,
verifiable `<= t` / `< t` logic (cumulative sums and `shift()`), never the full-history shortcut.

To keep this auditable rather than relying on the proof alone, **TMP1/TMP2 below are implemented
using the explicit, boundary-respecting cumulative-minimum method** (`groupby().cummin()` on data
sorted by `[address, Time step]`), and the full-history shortcut is used only in a **separate
validation cell** that empirically confirms the two methods agree -- demonstrating the proof
rather than depending on it in production code.

## 1. Imports and configuration

In [1]:
import time
import os
import gc
import numpy as np
import pandas as pd
import psutil
from collections import defaultdict

t0 = time.time()
def log(msg):
    print(f"[{time.time()-t0:7.1f}s] {msg}", flush=True)

def mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"
OUT_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\processed"
OUT_PATH = os.path.join(OUT_DIR, "transaction_time_safe_features_v1.parquet")

print("Setup complete.")

Setup complete.


## 2. Load transaction data (narrow column set -- not all 184 columns)

In [2]:
tx_cols = [
    "txId", "Time step",
    "in_BTC_max", "in_BTC_total",
    "out_BTC_max", "out_BTC_total",
    "fees", "total_BTC", "size",
    "num_input_addresses", "num_output_addresses",
]
tx_features = pd.read_csv(DATA_DIR + r"\txs_features.csv", usecols=tx_cols)
n_tx_total = len(tx_features)
print("tx_features shape:", tx_features.shape, " (expected 203769 rows)")
assert n_tx_total == 203769

tx_time = tx_features[["txId", "Time step"]].copy()

tx_features shape: (203769, 11)  (expected 203769 rows)


## 3. Load address edges, attach Time step (many-to-one merge -- cannot multiply rows)

In [3]:
addr_tx_raw = pd.read_csv(DATA_DIR + r"\AddrTx_edgelist.csv")
tx_addr_raw = pd.read_csv(DATA_DIR + r"\TxAddr_edgelist.csv")
print("addr_tx_raw:", addr_tx_raw.shape, " (expected 477117)")
print("tx_addr_raw:", tx_addr_raw.shape, " (expected 837124)")
assert addr_tx_raw.shape[0] == 477117
assert tx_addr_raw.shape[0] == 837124

addr_tx_raw = addr_tx_raw.rename(columns={"input_address": "address"})
tx_addr_raw = tx_addr_raw.rename(columns={"output_address": "address"})

addr_tx_time = addr_tx_raw.merge(tx_time, on="txId", how="left")
tx_addr_time = tx_addr_raw.merge(tx_time, on="txId", how="left")
print("addr_tx_time:", addr_tx_time.shape, " tx_addr_time:", tx_addr_time.shape)

assert addr_tx_time.shape[0] == 477117, "merge multiplied rows -- should never happen (many-to-one)"
assert tx_addr_time.shape[0] == 837124
assert addr_tx_time["Time step"].isna().sum() == 0
assert tx_addr_time["Time step"].isna().sum() == 0

del addr_tx_raw, tx_addr_raw
gc.collect()
print("Integrity checks passed.")

addr_tx_raw: (477117, 2)  (expected 477117)
tx_addr_raw: (837124, 2)  (expected 837124)
addr_tx_time: (477117, 3)  tx_addr_time: (837124, 3)
Integrity checks passed.


## 4. Duplicate audit (measure first, decide policy from real numbers)

Per project rule: never silently deduplicate. Policy stated explicitly: for every one of the 14
features, what matters is "was this address involved in this transaction" (binary), never how
many raw UTXO rows represent it -- so `(address, txId)` is deduplicated regardless of what the
count turns out to be, with the exact count reported here.

In [4]:
addr_tx_dupe_count = addr_tx_time.duplicated(subset=["address", "txId"]).sum()
tx_addr_dupe_count = tx_addr_time.duplicated(subset=["address", "txId"]).sum()
print(f"addr_tx_time duplicate (address, txId) rows: {addr_tx_dupe_count} / {len(addr_tx_time)}")
print(f"tx_addr_time duplicate (address, txId) rows: {tx_addr_dupe_count} / {len(tx_addr_time)}")

addr_tx_dedup = addr_tx_time.drop_duplicates(subset=["address", "txId"]).copy()
tx_addr_dedup = tx_addr_time.drop_duplicates(subset=["address", "txId"]).copy()
print("After dedup: addr_tx_dedup:", addr_tx_dedup.shape, " tx_addr_dedup:", tx_addr_dedup.shape)

addr_tx_time duplicate (address, txId) rows: 0 / 477117
tx_addr_time duplicate (address, txId) rows: 0 / 837124
After dedup: addr_tx_dedup: (477117, 3)  tx_addr_dedup: (837124, 3)


**Result: zero duplicates found in either file.** The dedup step is a documented no-op on this data -- the policy still applies going forward should the raw files ever change.

## 5. Transaction-intrinsic features: FIN3, FIN4, FIN5, FIN6, STR1 (no joins)

In [5]:
def safe_div(numer, denom):
    numer = numer.to_numpy(dtype=np.float64)
    denom = denom.to_numpy(dtype=np.float64)
    return np.where(denom == 0, 0.0, numer / denom)

features = pd.DataFrame({"txId": tx_features["txId"], "Time step": tx_features["Time step"]})

features["FIN3"] = safe_div(tx_features["in_BTC_max"], tx_features["in_BTC_total"])
features["FIN4"] = safe_div(tx_features["out_BTC_max"], tx_features["out_BTC_total"])
features["FIN5"] = safe_div(tx_features["fees"], tx_features["total_BTC"])
features["FIN6"] = safe_div(tx_features["fees"], tx_features["size"])
features["STR1"] = safe_div(tx_features["num_input_addresses"], tx_features["num_output_addresses"])

for col in ["FIN3", "FIN4", "FIN5", "FIN6", "STR1"]:
    n_inf = np.isinf(features[col]).sum()
    n_nan = features[col].isna().sum()
    print(f"{col}: inf={n_inf} nan={n_nan} min={features[col].min():.6f} max={features[col].max():.6f}")
    assert n_inf == 0, f"{col} contains inf -- division-by-zero guard failed"

FIN3: inf=0 nan=965 min=0.002716 max=1.000000
FIN4: inf=0 nan=965 min=0.002008 max=1.000000
FIN5: inf=0 nan=965 min=0.000000 max=44.825000
FIN6: inf=0 nan=965 min=0.000000 max=0.000226
STR1: inf=0 nan=965 min=0.000076 max=651.000000


## 6. STR2 -- same-transaction address reuse

Boundary: compares addresses **only within the same `txId`** -- an inner-merge on `[txId, address]`
makes it mechanically impossible to match an input address of transaction X against an output
address of transaction Y. No global address comparison ever happens.

In [6]:
reuse = addr_tx_dedup[["txId", "address"]].merge(
    tx_addr_dedup[["txId", "address"]], on=["txId", "address"], how="inner"
)
reused_txids = set(reuse["txId"].unique())
print(f"Transactions with input/output address reuse: {len(reused_txids)}")
features["STR2"] = features["txId"].isin(reused_txids).astype("int8")

Transactions with input/output address reuse: 45842


## 7. TMP1 / TMP2 -- address age at t

**Definition (corrected after review):** *age of the address based on any observed transaction
activity up to time t* -- input OR output role, not role-restricted. Boundary: `<= t`, using the
address's earliest recorded activity in **either role**.

### Correction applied

The first implementation of this notebook computed TMP1's age lookup from `addr_tx_dedup` alone
(input-role activity only) and TMP2's from `tx_addr_dedup` alone (output-role only). This was
**wrong**: the inherited lifetime features being replaced (`Local_feature_53/54/55`) are based on
an address's broader lifetime, not a role-specific one.

**Concrete example that caught the bug:** address A appears as an **output** at Time step 2, then
as an **input** at Time step 7. For a transaction at t=7 using A as an input, TMP1 must be
`7 - 2 = 5` (broader lifetime) -- the original implementation gave `7 - 7 = 0`, because it only
looked at A's input-role history and never saw its earlier output-role appearance.

**Fix:** build one combined `(address, Time step)` activity table from **both** edge files first,
compute the earliest-activity lookup once from that combined table, then apply it to input
addresses (for TMP1) and output addresses (for TMP2) separately -- only the *set of addresses
being looked up* differs between TMP1 and TMP2; the *age source* is now identical and role-
agnostic for both, as required.

The explicit, boundary-respecting cumulative-minimum method is still used (not the full-history
shortcut) -- `groupby().cummin()` on data sorted by `[address, Time step]`, `kind="mergesort"` for
determinism.

In [7]:
combined_activity_age = pd.concat([
    addr_tx_dedup[["address", "Time step"]],
    tx_addr_dedup[["address", "Time step"]],
], ignore_index=True).drop_duplicates(subset=["address", "Time step"])

combined_activity_age = combined_activity_age.sort_values(
    by=["address", "Time step"], kind="mergesort"
).reset_index(drop=True)
combined_activity_age["first_seen_leq_t"] = combined_activity_age.groupby(
    "address", sort=False
)["Time step"].cummin()

age_lookup = combined_activity_age[["address", "Time step", "first_seen_leq_t"]]

# Attach the SAME combined-activity age lookup to both input-role and
# output-role address occurrences (many-to-one merge on [address, Time step])
addr_tx_age = addr_tx_dedup.merge(age_lookup, on=["address", "Time step"], how="left")
addr_tx_age["age"] = addr_tx_age["Time step"] - addr_tx_age["first_seen_leq_t"]
assert (addr_tx_age["age"] >= 0).all()

tx_addr_age = tx_addr_dedup.merge(age_lookup, on=["address", "Time step"], how="left")
tx_addr_age["age"] = tx_addr_age["Time step"] - tx_addr_age["first_seen_leq_t"]
assert (tx_addr_age["age"] >= 0).all()

assert addr_tx_age.shape[0] == addr_tx_dedup.shape[0]
assert tx_addr_age.shape[0] == tx_addr_dedup.shape[0]
print("TMP1/TMP2 per-address-role age computed (combined-role lookup) and boundary-asserted (>=0).")

TMP1/TMP2 per-address-role age computed (combined-role lookup) and boundary-asserted (>=0).


### Validation: proving the full-history MIN shortcut is equivalent (demonstration only)

Re-run on the corrected **combined-role** activity table -- the proof still holds (it depends only
on `t` being a member of the activity set being minimized, regardless of which roles are pooled
into that set). This does not replace the explicit `<= t` computation above -- it independently
confirms the proof holds on this real data, as supporting evidence.

In [8]:
combined_all = pd.concat([
    addr_tx_dedup[["address", "Time step"]],
    tx_addr_dedup[["address", "Time step"]],
], ignore_index=True)
global_min_combined = combined_all.groupby("address")["Time step"].transform("min")
naive_age_check = combined_all.copy()
naive_age_check["naive_age"] = naive_age_check["Time step"] - global_min_combined
naive_age_check = naive_age_check.drop_duplicates(subset=["address", "Time step"])
check = naive_age_check.merge(age_lookup, on=["address", "Time step"], how="left")
check["explicit_age"] = check["Time step"] - check["first_seen_leq_t"]
mismatch = (check["naive_age"].values != check["explicit_age"].values).sum()
print(f"Mismatches between global-MIN shortcut and explicit <=t cummin (combined roles): {mismatch}")
assert mismatch == 0, "global-min proof does not hold empirically -- do not use the shortcut"
print("Proof empirically confirmed on the real dataset (combined-role activity): 0 mismatches.")

Mismatches between global-MIN shortcut and explicit <=t cummin (combined roles): 0
Proof empirically confirmed on the real dataset (combined-role activity): 0 mismatches.


In [9]:
# Collapse to one value per transaction: MIN across a transaction's addresses
# (deliberate forensic choice -- see introduction, not a mathematical necessity)
tmp1_per_tx = addr_tx_age.groupby("txId")["age"].min().rename("TMP1")
tmp2_per_tx = tx_addr_age.groupby("txId")["age"].min().rename("TMP2")

features = features.merge(tmp1_per_tx, on="txId", how="left")
features = features.merge(tmp2_per_tx, on="txId", how="left")
print("TMP1 missing (tx with no recorded input addresses):", features["TMP1"].isna().sum())
print("TMP2 missing (tx with no recorded output addresses):", features["TMP2"].isna().sum())

TMP1 missing (tx with no recorded input addresses): 965
TMP2 missing (tx with no recorded output addresses): 965


**Impact of the fix (measured against the pre-correction version of this notebook):**

| | Changed | Unchanged | Direction |
|---|---|---|---|
| TMP1 | 3,174 / 203,769 (1.56%) | 199,630 (of non-null pairs) | 100% increases, 0 decreases |
| TMP2 | 688 / 203,769 (0.34%) | 202,116 (of non-null pairs) | 100% increases, 0 decreases |

Every change is an *increase*, never a decrease -- expected and reassuring, since pooling in
additional (output-role, for TMP1) history can only push an address's earliest-seen time earlier
or leave it unchanged, never later. Missing-value counts (965 for both) are unaffected, since the
fix changes *which* activity is consulted, not *whether* any activity exists at all.

## 8. TMP4 / TMP5 -- prior transaction count as sender/receiver, self-excluded

**Unchanged by this correction** -- these are deliberately role-specific by definition (sender
count and receiver count are separate concepts on purpose, per Part 2 of the feature-selection
discussion), unlike TMP1/TMP2's age which is meant to reflect broader lifetime.

Boundary: `<= t`, count of *other* transactions (self always subtracted). Algorithm: per-
`(address, Time step)` bucket counts, sorted, cumulative-summed within each address, merged back
by `[address, Time step]` (many-to-one, safe), then `-1` for self-exclusion -- valid because every
row being annotated is, by construction, itself a member of its own `(address, Time step)` bucket.

In [10]:
def cumulative_count_minus_self(df, addr_col="address"):
    per_bucket = df.groupby([addr_col, "Time step"]).size().rename("bucket_count").reset_index()
    per_bucket = per_bucket.sort_values(by=[addr_col, "Time step"], kind="mergesort")
    per_bucket["cum_count"] = per_bucket.groupby(addr_col, sort=False)["bucket_count"].cumsum()
    merged = df.merge(per_bucket[[addr_col, "Time step", "cum_count"]], on=[addr_col, "Time step"], how="left")
    merged["prior_count"] = merged["cum_count"] - 1
    assert (merged["prior_count"] >= 0).all()
    return merged

sender_counted = cumulative_count_minus_self(addr_tx_dedup, "address")
receiver_counted = cumulative_count_minus_self(tx_addr_dedup, "address")

tmp4_per_tx = sender_counted.groupby("txId")["prior_count"].min().rename("TMP4")
tmp5_per_tx = receiver_counted.groupby("txId")["prior_count"].min().rename("TMP5")

features = features.merge(tmp4_per_tx, on="txId", how="left")
features = features.merge(tmp5_per_tx, on="txId", how="left")
print("TMP4 missing:", features["TMP4"].isna().sum(), " TMP5 missing:", features["TMP5"].isna().sum())

TMP4 missing: 965  TMP5 missing: 965


## 9. TMP6 / TMP7 -- recency (strictly `< t`) and distinct-time-step breadth (`<= t`)

Built from a **combined** sender+receiver activity table (an address's overall activity, not
role-specific) -- deduplicated to unique `(address, Time step)` pairs, sorted deterministically.
`shift(1)` on this sorted series is mechanically restricted to a strictly-prior distinct time
step -- it cannot reference the current row or a future one, by construction, not by a runtime
filter that could be implemented incorrectly.

In [11]:
combined_activity = pd.concat([
    addr_tx_dedup[["address", "Time step"]],
    tx_addr_dedup[["address", "Time step"]],
], ignore_index=True).drop_duplicates(subset=["address", "Time step"])

combined_activity = combined_activity.sort_values(
    by=["address", "Time step"], kind="mergesort"
).reset_index(drop=True)

combined_activity["prev_time_step"] = combined_activity.groupby("address", sort=False)["Time step"].shift(1)
combined_activity["TMP6_gap"] = combined_activity["Time step"] - combined_activity["prev_time_step"]
combined_activity["is_first_appearance"] = combined_activity["prev_time_step"].isna()

valid_gap_rows = combined_activity[~combined_activity["is_first_appearance"]]
assert (valid_gap_rows["prev_time_step"] < valid_gap_rows["Time step"]).all(), "TMP6 boundary violated"
print("TMP6 strict <t boundary assertion passed on", len(valid_gap_rows), "rows with a defined gap.")

combined_activity["TMP7_rank"] = combined_activity.groupby("address", sort=False).cumcount() + 1

tmp6_lookup = combined_activity[["address", "Time step", "TMP6_gap", "is_first_appearance"]]
tmp7_lookup = combined_activity[["address", "Time step", "TMP7_rank"]]

# TMP6/TMP7 are defined over INPUT addresses (matches TMP1/TMP4's role)
addr_tx_t67 = addr_tx_dedup.merge(tmp6_lookup, on=["address", "Time step"], how="left") \
                            .merge(tmp7_lookup, on=["address", "Time step"], how="left")

tmp6_per_tx = addr_tx_t67.groupby("txId")["TMP6_gap"].min().rename("TMP6")
tmp7_per_tx = addr_tx_t67.groupby("txId")["TMP7_rank"].min().rename("TMP7")

features = features.merge(tmp6_per_tx, on="txId", how="left")
features = features.merge(tmp7_per_tx, on="txId", how="left")

features["TMP6_is_first_appearance"] = features["TMP6"].isna().astype("int8")

print("TMP6 missing (= first-appearance flag = 1):", features["TMP6"].isna().sum())
print("TMP7 missing:", features["TMP7"].isna().sum())

TMP6 strict <t boundary assertion passed on 31514 rows with a defined gap.
TMP6 missing (= first-appearance flag = 1): 172255
TMP7 missing: 965


In [12]:
# Plausibility cross-check: TMP6 and TMP7 come from the same combined_activity
# table -- TMP6_gap is NaN exactly when TMP7_rank == 1 (per row). At the
# transaction level, TMP6 can only be NaN if EVERY input address has rank 1,
# which forces the transaction's minimum TMP7 to also be 1.
tmp6_nan_mask = features["TMP6"].isna() & features["TMP7"].notna()
assert (features.loc[tmp6_nan_mask, "TMP7"] == 1).all(), \
    "TMP6 is NaN for a transaction whose minimum TMP7 != 1 -- logic error"
print("Plausibility cross-check (TMP6 NaN implies TMP7==1) passed.")

Plausibility cross-check (TMP6 NaN implies TMP7==1) passed.


## 10. Pre-flight stats before NET1/NET2 (safety check before expensive processing)

Per explicit instruction: if any single address's activity implies a runaway cost, stop and
report -- do not silently switch to a different definition or grind for hours.

In [13]:
n_unique_addr_in = addr_tx_dedup["address"].nunique()
n_unique_addr_out = tx_addr_dedup["address"].nunique()
n_unique_txid = features["txId"].nunique()
max_tx_per_addr_in = addr_tx_dedup.groupby("address").size().max()
max_tx_per_addr_out = tx_addr_dedup.groupby("address").size().max()

print("=" * 60)
print("PRE-FLIGHT STATS BEFORE NET1/NET2")
print("=" * 60)
print(f"Unique input addresses     : {n_unique_addr_in}")
print(f"Unique output addresses    : {n_unique_addr_out}")
print(f"Unique txId                : {n_unique_txid}")
print(f"Max transactions on ONE input address : {max_tx_per_addr_in}")
print(f"Max transactions on ONE output address: {max_tx_per_addr_out}")
print(f"Process RSS memory before NET step    : {mem_mb():.1f} MB")
print("=" * 60)

MAX_REASONABLE_DEGREE = 5000
if max_tx_per_addr_in > MAX_REASONABLE_DEGREE or max_tx_per_addr_out > MAX_REASONABLE_DEGREE:
    raise RuntimeError(
        f"An address exceeds the {MAX_REASONABLE_DEGREE}-transaction safety threshold "
        f"(in={max_tx_per_addr_in}, out={max_tx_per_addr_out}). Stopping before NET1/NET2 "
        f"to avoid an uncontrolled runtime blowup -- investigate before proceeding."
    )
print("Safety threshold OK -- proceeding with NET1/NET2.")

PRE-FLIGHT STATS BEFORE NET1/NET2
Unique input addresses     : 400212
Unique output addresses    : 641043
Unique txId                : 203769
Max transactions on ONE input address : 1453
Max transactions on ONE output address: 1471
Process RSS memory before NET step    : 222.8 MB
Safety threshold OK -- proceeding with NET1/NET2.


## 11. NET1 / NET2 -- historical address-linked transaction count

**Rejected approach:** a full self-join of the address-edge table on `address` (would find
transaction pairs sharing an address) -- this is exactly the many-to-many pattern that caused the
earlier 6GB memory incident, since a high-reuse "hub" address produces a quadratic number of
pairs.

**Chosen approach:** per-address grouped transaction lists (plain Python dicts, size bounded by
the raw edge-file row counts, never a Cartesian product) + a per-transaction set union across that
transaction's own addresses, restricted to `Time step <= t` and explicitly excluding the current
`txId`. Progress logged every 20,000 transactions.

In [14]:
addr_to_txns_in = defaultdict(list)
for addr, ts, txid in zip(addr_tx_dedup["address"].values, addr_tx_dedup["Time step"].values, addr_tx_dedup["txId"].values):
    addr_to_txns_in[addr].append((ts, txid))
print(f"addr_to_txns_in built: {len(addr_to_txns_in)} unique addresses. Memory: {mem_mb():.1f} MB")

addr_to_txns_out = defaultdict(list)
for addr, ts, txid in zip(tx_addr_dedup["address"].values, tx_addr_dedup["Time step"].values, tx_addr_dedup["txId"].values):
    addr_to_txns_out[addr].append((ts, txid))
print(f"addr_to_txns_out built: {len(addr_to_txns_out)} unique addresses. Memory: {mem_mb():.1f} MB")

tx_to_input_addrs = addr_tx_dedup.groupby("txId")["address"].apply(list).to_dict()
tx_to_output_addrs = tx_addr_dedup.groupby("txId")["address"].apply(list).to_dict()
tx_to_time = dict(zip(features["txId"].values, features["Time step"].values))
print(f"Lookup dicts built. Memory: {mem_mb():.1f} MB")

addr_to_txns_in built: 400212 unique addresses. Memory: 338.3 MB
addr_to_txns_out built: 641043 unique addresses. Memory: 520.3 MB
Lookup dicts built. Memory: 613.8 MB


In [15]:
def compute_net(all_tx_ids, tx_to_addrs, addr_to_txns, label):
    log(f"Computing {label} for {len(all_tx_ids)} transactions...")
    results = {}
    progress_interval = 20000
    for i, txid in enumerate(all_tx_ids):
        t = tx_to_time[txid]
        addrs = tx_to_addrs.get(txid, [])
        if not addrs:
            results[txid] = np.nan
            continue
        neighbours = set()
        for a in addrs:
            for (ts, other_txid) in addr_to_txns.get(a, ()):
                if ts <= t and other_txid != txid:
                    neighbours.add(other_txid)
        results[txid] = len(neighbours)
        if (i + 1) % progress_interval == 0:
            log(f"  {label}: {i+1}/{len(all_tx_ids)} processed. Memory: {mem_mb():.1f} MB")
    log(f"{label} complete.")
    return results

all_tx_ids = features["txId"].tolist()
net1_dict = compute_net(all_tx_ids, tx_to_input_addrs, addr_to_txns_in, "NET1")
net2_dict = compute_net(all_tx_ids, tx_to_output_addrs, addr_to_txns_out, "NET2")

features["NET1"] = features["txId"].map(net1_dict)
features["NET2"] = features["txId"].map(net2_dict)

print("NET1 missing:", features["NET1"].isna().sum(), " NET2 missing:", features["NET2"].isna().sum())
print(f"NET1 range: [{features['NET1'].min()}, {features['NET1'].max()}]")
print(f"NET2 range: [{features['NET2'].min()}, {features['NET2'].max()}]")

[    8.9s] Computing NET1 for 203769 transactions...
[    8.9s]   NET1: 20000/203769 processed. Memory: 621.6 MB
[    9.0s]   NET1: 40000/203769 processed. Memory: 622.9 MB
[    9.2s]   NET1: 60000/203769 processed. Memory: 624.2 MB
[    9.4s]   NET1: 80000/203769 processed. Memory: 624.2 MB
[    9.6s]   NET1: 100000/203769 processed. Memory: 626.8 MB
[    9.8s]   NET1: 120000/203769 processed. Memory: 626.8 MB
[   10.0s]   NET1: 140000/203769 processed. Memory: 626.8 MB
[   10.2s]   NET1: 160000/203769 processed. Memory: 626.9 MB
[   10.3s]   NET1: 180000/203769 processed. Memory: 631.9 MB
[   10.5s]   NET1: 200000/203769 processed. Memory: 631.9 MB
[   10.5s] NET1 complete.
[   10.5s] Computing NET2 for 203769 transactions...
[   10.7s]   NET2: 20000/203769 processed. Memory: 631.9 MB
[   10.9s]   NET2: 40000/203769 processed. Memory: 633.2 MB
[   11.1s]   NET2: 60000/203769 processed. Memory: 634.5 MB
[   11.3s]   NET2: 80000/203769 processed. Memory: 634.6 MB
[   11.6s]   NET2: 100

In [16]:
sample_check_ids = features["txId"].sample(n=200, random_state=42).tolist()
for txid in sample_check_ids:
    t = tx_to_time[txid]
    addrs = tx_to_input_addrs.get(txid, [])
    neighbours = set()
    for a in addrs:
        for (ts, other_txid) in addr_to_txns_in.get(a, ()):
            if ts <= t and other_txid != txid:
                neighbours.add(other_txid)
    assert txid not in neighbours, f"Self-exclusion failed for {txid}"
print("Self-exclusion spot-check (200 random transactions) passed.")

Self-exclusion spot-check (200 random transactions) passed.


## 12. Leakage validation suite (concrete assertions, not a NaN check)

In [17]:
feature_cols = ["FIN3", "FIN4", "FIN5", "FIN6", "STR1", "STR2", "TMP1", "TMP2", "TMP4", "TMP5", "TMP6", "TMP7", "NET1", "NET2"]

assert len(features) == 203769, f"row count {len(features)} != 203769"
print("[PASS] Exactly 203,769 final transactions.")

assert features["txId"].is_unique
print("[PASS] txId is unique.")

assert features["txId"].nunique() == len(features)
print("[PASS] No row multiplication (nunique == len).")

for col in feature_cols:
    n_inf = np.isinf(features[col].astype("float64")).sum()
    assert n_inf == 0, f"{col} contains inf"
print("[PASS] No inf values in any feature.")

print("\nMissing-value report (not silently imputed):")
missing_report = features[feature_cols].isna().sum().rename("missing_count").to_frame()
missing_report["missing_pct"] = (missing_report["missing_count"] / len(features) * 100).round(2)
print(missing_report.to_string())

[PASS] Exactly 203,769 final transactions.
[PASS] txId is unique.
[PASS] No row multiplication (nunique == len).
[PASS] No inf values in any feature.

Missing-value report (not silently imputed):
      missing_count  missing_pct
FIN3            965         0.47
FIN4            965         0.47
FIN5            965         0.47
FIN6            965         0.47
STR1            965         0.47
STR2              0         0.00
TMP1            965         0.47
TMP2            965         0.47
TMP4            965         0.47
TMP5            965         0.47
TMP6         172255        84.53
TMP7            965         0.47
NET1            965         0.47
NET2            965         0.47


**Missingness unchanged by the TMP1/TMP2 correction** -- 965 for both, identical to before. "
   "This is expected: the fix changed *which* activity feeds the age lookup, not *whether* a "
   "transaction has any recorded addresses at all. The 965 rows are transactions absent from "
   "both edge files entirely (same rows the original notebook's augmented-features audit found "
   "missing `in_BTC_*`/`out_BTC_*`).

In [18]:
assert features["TMP1"].dropna().between(0, 48).all(), "TMP1 out of [0,48]"
assert features["TMP2"].dropna().between(0, 48).all(), "TMP2 out of [0,48]"
print("[PASS] TMP1, TMP2 in [0, 48].")

assert (features["TMP4"].dropna() >= 0).all()
assert (features["TMP5"].dropna() >= 0).all()
assert (features["TMP7"].dropna() >= 0).all()
print("[PASS] TMP4, TMP5, TMP7 >= 0.")

assert (features["TMP6"].dropna() > 0).all(), "TMP6 must be a strictly positive gap where defined"
print("[PASS] TMP6 is either a strictly positive gap or NaN (first-appearance).")

assert (features["NET1"].dropna() >= 0).all()
assert (features["NET2"].dropna() >= 0).all()
print("[PASS] NET1, NET2 >= 0.")

assert features["STR2"].dropna().isin([0, 1]).all()
print("[PASS] STR2 is strictly binary.")

[PASS] TMP1, TMP2 in [0, 48].
[PASS] TMP4, TMP5, TMP7 >= 0.
[PASS] TMP6 is either a strictly positive gap or NaN (first-appearance).
[PASS] NET1, NET2 >= 0.
[PASS] STR2 is strictly binary.


In [19]:
addr_tx_check = pd.read_csv(DATA_DIR + r"\AddrTx_edgelist.csv")
tx_addr_check = pd.read_csv(DATA_DIR + r"\TxAddr_edgelist.csv")
assert addr_tx_check.shape[0] == 477117
assert tx_addr_check.shape[0] == 837124
assert addr_tx_check["input_address"].nunique() == 400212
assert tx_addr_check["output_address"].nunique() == 641043
print("[PASS] Address/edge counts match raw files exactly (477117, 837124, 400212, 641043).")
del addr_tx_check, tx_addr_check
gc.collect()

[PASS] Address/edge counts match raw files exactly (477117, 837124, 400212, 641043).


## 13. Distribution validation

In [20]:
print(features[feature_cols].describe().T.to_string())

         count       mean         std       min           25%           50%        75%          max
FIN3  202804.0   0.916328    0.200416  0.002716  1.000000e+00  1.000000e+00   1.000000     1.000000
FIN4  202804.0   0.881011    0.167313  0.002008  8.114732e-01  9.697273e-01   0.998177     1.000000
FIN5  202804.0   0.027090    0.391850  0.000000  5.123311e-05  4.527098e-04   0.004114    44.825000
FIN6  202804.0   0.000002    0.000007  0.000000  4.444444e-07  9.967161e-07   0.000002     0.000226
STR1  202804.0   1.441705   10.130654  0.000076  5.000000e-01  5.000000e-01   1.000000   651.000000
STR2  203769.0   0.224970    0.417564  0.000000  0.000000e+00  0.000000e+00   0.000000     1.000000
TMP1  202804.0   1.305734    5.178283  0.000000  0.000000e+00  0.000000e+00   0.000000    48.000000
TMP2  202804.0   0.266030    2.301776  0.000000  0.000000e+00  0.000000e+00   0.000000    46.000000
TMP4  202804.0  36.089352  133.901328  0.000000  0.000000e+00  0.000000e+00   0.000000  1452.000000


**TMP1 mean increased from 1.178 to 1.306, TMP2 mean from 0.254 to 0.266** -- small but "
   "real upward shifts, consistent with the fix only ever increasing ages (Section 7). Max "
   "values unchanged (0 and 48/46 respectively -- the transactions already at the boundary "
   "were already using their true earliest activity in one role or the other).

## 14. Correlation / redundancy audit (re-run after the TMP1/TMP2 fix)

Flags exactly what Part 2's feature-selection discussion anticipated (FIN3/STR1, TMP1/TMP7) plus
some genuine surprises worth documenting honestly rather than hiding.

In [21]:
corr = features[feature_cols].corr(method="spearman")
print(corr.round(3).to_string())

high_corr_pairs = []
for i, c1 in enumerate(feature_cols):
    for c2 in feature_cols[i+1:]:
        v = corr.loc[c1, c2]
        if abs(v) >= 0.7:
            high_corr_pairs.append((c1, c2, round(v, 3)))
print("\nPairs with |Spearman| >= 0.7:")
print(high_corr_pairs if high_corr_pairs else "None")


Pairs with |Spearman| >= 0.7:
[('FIN3', 'STR1', -0.718), ('STR2', 'TMP4', 0.795), ('STR2', 'NET1', 0.737), ('TMP1', 'TMP7', 0.999), ('TMP4', 'NET1', 0.9)]


       FIN3   FIN4   FIN5   FIN6   STR1   STR2   TMP1   TMP2   TMP4   TMP5   TMP6   TMP7   NET1   NET2
FIN3  1.000  0.183 -0.233  0.195 -0.718  0.146  0.106 -0.009  0.209  0.035 -0.141  0.107  0.009  0.060
FIN4  0.183  1.000 -0.257  0.052  0.186  0.120  0.083  0.175  0.153  0.282 -0.092  0.085  0.121  0.152
FIN5 -0.233 -0.257  1.000  0.201  0.290 -0.203 -0.162  0.034 -0.243  0.052  0.317 -0.168 -0.220 -0.234
FIN6  0.195  0.052  0.201  1.000 -0.127 -0.031  0.058 -0.028  0.026 -0.029  0.028  0.057 -0.017 -0.089
STR1 -0.718  0.186  0.290 -0.127  1.000 -0.139 -0.114  0.122 -0.188  0.192  0.140 -0.117 -0.035 -0.104
STR2  0.146  0.120 -0.203 -0.031 -0.139  1.000  0.431  0.036  0.795  0.175 -0.266  0.437  0.737  0.591
TMP1  0.106  0.083 -0.162  0.058 -0.114  0.431  1.000  0.123  0.634  0.063  0.020  0.999  0.583  0.431
TMP2 -0.009  0.175  0.034 -0.028  0.122  0.036  0.123  1.000  0.059  0.486 -0.035  0.125  0.079  0.208
TMP4  0.209  0.153 -0.243  0.026 -0.188  0.795  0.634  0.059  1.000  0.19

**Important change from the pre-fix version: TMP1/TMP7 correlation rose from 0.955 to "
   "0.999 -- effectively perfect now.** This makes sense in hindsight: once TMP1 correctly uses "
   "combined-role first-activity (the same source table TMP7 already used), both features are "
   "now derived from the *same* underlying combined-activity ranking -- TMP1 is the time-gap "
   "version, TMP7 is the count-of-distinct-steps version, of the same signal. **This is now "
   "flagged as a real, near-total redundancy, not a minor overlap** -- worth deciding before the "
   "GNN stage whether to keep both TMP1 and TMP7, since one is now almost fully predictable from "
   "the other on this dataset. Not resolved here -- surfaced for your decision, since Part 2's "
   "feature set was explicitly approved and any drop needs the same explicit approval.

   TMP4/NET1 (0.900) and STR2/TMP4 (0.795) are essentially unchanged from before (0.900, 0.795), "
   "as expected since neither depends on TMP1/TMP2.

## 15. Random manual verification -- 5 real transactions, hand-traced independently

**Updated per review:** TMP1/TMP2 are now independently recomputed using the **combined**
address-activity table (both roles), matching the corrected production logic -- not the old
role-restricted method. The sample deliberately includes 3 transactions known to have changed
under the fix (drawn from the actual before/after diff) plus 2 random transactions, so this check
exercises the fix directly rather than coincidentally avoiding it.

In [22]:
combined_manual = pd.concat([
    addr_tx_dedup[["address", "Time step"]],
    tx_addr_dedup[["address", "Time step"]],
], ignore_index=True)

# Deliberately include known-CHANGED transactions (from the actual before/after
# diff against the pre-correction pipeline) so this check exercises the fix.
old_features = pd.read_pickle("ts_safe_features_full_OLD_before_tmp1tmp2_fix.pkl")  # pre-correction snapshot
changed_cmp = old_features[["txId", "TMP1"]].merge(features[["txId", "TMP1"]], on="txId", suffixes=("_old", "_new"))
both_notna = changed_cmp["TMP1_old"].notna() & changed_cmp["TMP1_new"].notna()
truly_changed = both_notna & (changed_cmp["TMP1_old"] != changed_cmp["TMP1_new"])
known_changed_ids = changed_cmp.loc[truly_changed, "txId"].sample(3, random_state=1).tolist()
random_ids = features[features["TMP1"].notna()]["txId"].sample(2, random_state=7).tolist()
sample_ids = known_changed_ids + random_ids

for txid in sample_ids:
    row = features[features["txId"] == txid].iloc[0]
    t = row["Time step"]
    in_addrs = addr_tx_dedup.loc[addr_tx_dedup["txId"] == txid, "address"].tolist()
    out_addrs = tx_addr_dedup.loc[tx_addr_dedup["txId"] == txid, "address"].tolist()

    manual_ages = []
    for a in in_addrs:
        first_seen = combined_manual.loc[
            (combined_manual["address"] == a) & (combined_manual["Time step"] <= t), "Time step"
        ].min()
        manual_ages.append(t - first_seen)
    manual_tmp1 = min(manual_ages) if manual_ages else np.nan

    manual_ages2 = []
    for a in out_addrs:
        first_seen = combined_manual.loc[
            (combined_manual["address"] == a) & (combined_manual["Time step"] <= t), "Time step"
        ].min()
        manual_ages2.append(t - first_seen)
    manual_tmp2 = min(manual_ages2) if manual_ages2 else np.nan

    manual_tmp4s = []
    for a in in_addrs:
        prior = addr_tx_dedup.loc[
            (addr_tx_dedup["address"] == a) & (addr_tx_dedup["Time step"] <= t) & (addr_tx_dedup["txId"] != txid)
        ]
        manual_tmp4s.append(len(prior))
    manual_tmp4 = min(manual_tmp4s) if manual_tmp4s else np.nan

    manual_neighbours = set()
    for a in in_addrs:
        other = addr_tx_dedup.loc[
            (addr_tx_dedup["address"] == a) & (addr_tx_dedup["Time step"] <= t) & (addr_tx_dedup["txId"] != txid), "txId"
        ]
        manual_neighbours.update(other.tolist())
    manual_net1 = len(manual_neighbours)

    tag = "KNOWN-CHANGED" if txid in known_changed_ids else "random"
    print(f"\ntxId={txid}  Time step={t}  #input_addrs={len(in_addrs)}  #output_addrs={len(out_addrs)}  [{tag}]")
    print(f"  TMP1: pipeline={row['TMP1']}  manual={manual_tmp1}  match={row['TMP1']==manual_tmp1}")
    print(f"  TMP2: pipeline={row['TMP2']}  manual={manual_tmp2}  match={row['TMP2']==manual_tmp2}")
    print(f"  TMP4: pipeline={row['TMP4']}  manual={manual_tmp4}  match={row['TMP4']==manual_tmp4}")
    print(f"  NET1: pipeline={row['NET1']}  manual={manual_net1}  match={row['NET1']==manual_net1}")
    assert row["TMP1"] == manual_tmp1
    assert row["TMP2"] == manual_tmp2
    assert row["TMP4"] == manual_tmp4
    assert row["NET1"] == manual_net1

print("\n[PASS] All 5 manually-traced transactions match the pipeline exactly.")


txId=32032827  Time step=8.0  #input_addrs=1  #output_addrs=2  [KNOWN-CHANGED]
  TMP1: pipeline=6.0  manual=6.0  match=True
  TMP2: pipeline=0.0  manual=0.0  match=True
  TMP4: pipeline=0.0  manual=0  match=True
  NET1: pipeline=0.0  manual=0  match=True

txId=96279162  Time step=7.0  #input_addrs=1  #output_addrs=1  [KNOWN-CHANGED]
  TMP1: pipeline=5.0  manual=5.0  match=True
  TMP2: pipeline=0.0  manual=0.0  match=True
  TMP4: pipeline=0.0  manual=0  match=True
  NET1: pipeline=0.0  manual=0  match=True

txId=244312783  Time step=3.0  #input_addrs=1  #output_addrs=2  [KNOWN-CHANGED]
  TMP1: pipeline=2.0  manual=2.0  match=True
  TMP2: pipeline=0.0  manual=0.0  match=True
  TMP4: pipeline=11.0  manual=11  match=True
  NET1: pipeline=11.0  manual=11  match=True

txId=97423640  Time step=40.0  #input_addrs=3  #output_addrs=2  [random]
  TMP1: pipeline=0.0  manual=0.0  match=True
  TMP2: pipeline=0.0  manual=0.0  match=True
  TMP4: pipeline=0.0  manual=0  match=True
  NET1: pipeline=0.0

**`txId=96279162` is a live example of exactly the bug-report scenario:** Time step=7, "
   "single input address, TMP1=5.0 -- the address was active in an earlier role at step 2, "
   "input-use at step 7, age correctly reflects the 5-step gap rather than 0.

## 16. Final feature matrix: dtype casting and save

`txId` and `Time step` as regular columns (not an index), explicit dtype control (counts as
`Int32`, ages/gaps as `Int16` -- nullable, since 965 rows and TMP6's 172,255 rows are genuinely
missing, not zero), ratios as `float32`, `STR2` as `int8`. Saved as Parquet (exact dtype
preservation, unlike CSV round-tripping).

In [23]:
final = features.copy()
final["txId"] = final["txId"].astype("int64")
final["Time step"] = final["Time step"].astype("int16")
for col in ["FIN3", "FIN4", "FIN5", "FIN6", "STR1"]:
    final[col] = final[col].astype("float32")
final["STR2"] = final["STR2"].astype("int8")
for col in ["TMP1", "TMP2"]:
    final[col] = final[col].astype("Int16")
for col in ["TMP4", "TMP5", "TMP7", "NET1", "NET2"]:
    final[col] = final[col].astype("Int32")
final["TMP6"] = final["TMP6"].astype("Int16")
final["TMP6_is_first_appearance"] = final["TMP6_is_first_appearance"].astype("int8")

final = final[["txId", "Time step"] + feature_cols + ["TMP6_is_first_appearance"]]

os.makedirs(OUT_DIR, exist_ok=True)
final.to_parquet(OUT_PATH, index=False)
print("Saved to:", OUT_PATH)
print()
print(final.dtypes)

Saved to: C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\processed\transaction_time_safe_features_v1.parquet

txId                          int64
Time step                     int16
FIN3                        float32
FIN4                        float32
FIN5                        float32
FIN6                        float32
STR1                        float32
STR2                           int8
TMP1                          Int16
TMP2                          Int16
TMP4                          Int32
TMP5                          Int32
TMP6                          Int16
TMP7                          Int32
NET1                          Int32
NET2                          Int32
TMP6_is_first_appearance       int8
dtype: object


In [24]:
reloaded = pd.read_parquet(OUT_PATH)
assert reloaded.equals(final), "reload mismatch -- parquet round-trip not exact"
print("[PASS] Parquet round-trip is exact (deterministic, reproducible).")

del addr_tx_dedup, tx_addr_dedup, addr_tx_time, tx_addr_time
del addr_to_txns_in, addr_to_txns_out, tx_to_input_addrs, tx_to_output_addrs
gc.collect()
print(f"Memory after cleanup: {mem_mb():.1f} MB")

[PASS] Parquet round-trip is exact (deterministic, reproducible).
Memory after cleanup: 645.6 MB


## 17. Final audit table

For every feature: definition, family, exact safe boundary, source, whether it addresses one of
the 13/14 temporal-availability concerns from the prior audit, and its measured distribution.

In [25]:
audit_summary = pd.DataFrame([
    {"Feature": "FIN3", "Definition": "Largest input share = in_BTC_max/in_BTC_total", "Family": "Financial",
     "Safe_boundary": "None needed -- transaction-intrinsic", "Source": "txs_features.csv",
     "Addresses_temporal_concern": "No"},
    {"Feature": "FIN4", "Definition": "Largest output share = out_BTC_max/out_BTC_total", "Family": "Financial",
     "Safe_boundary": "None needed -- transaction-intrinsic", "Source": "txs_features.csv",
     "Addresses_temporal_concern": "No"},
    {"Feature": "FIN5", "Definition": "Fee-to-value ratio = fees/total_BTC", "Family": "Financial",
     "Safe_boundary": "None needed -- transaction-intrinsic", "Source": "txs_features.csv",
     "Addresses_temporal_concern": "No"},
    {"Feature": "FIN6", "Definition": "Fee rate = fees/size", "Family": "Financial",
     "Safe_boundary": "None needed -- transaction-intrinsic", "Source": "txs_features.csv",
     "Addresses_temporal_concern": "No"},
    {"Feature": "STR1", "Definition": "Input/output address count ratio", "Family": "Structural",
     "Safe_boundary": "None needed -- transaction-intrinsic", "Source": "txs_features.csv",
     "Addresses_temporal_concern": "No"},
    {"Feature": "STR2", "Definition": "Same-transaction address reuse flag", "Family": "Structural",
     "Safe_boundary": "Scoped to this txId only", "Source": "AddrTx+TxAddr edgelists",
     "Addresses_temporal_concern": "No"},
    {"Feature": "TMP1", "Definition": "Age of the (min) input address based on any observed transaction activity up to time t", "Family": "Temporal",
     "Safe_boundary": "<= t (explicit cummin, combined roles)", "Source": "AddrTx+TxAddr edgelists",
     "Addresses_temporal_concern": "Yes (53,54,55)"},
    {"Feature": "TMP2", "Definition": "Age of the (min) output address based on any observed transaction activity up to time t", "Family": "Temporal",
     "Safe_boundary": "<= t (explicit cummin, combined roles)", "Source": "AddrTx+TxAddr edgelists",
     "Addresses_temporal_concern": "Yes (90)"},
    {"Feature": "TMP4", "Definition": "Min prior sender-tx count, self-excluded", "Family": "Temporal",
     "Safe_boundary": "<= t (cumsum - 1)", "Source": "AddrTx_edgelist.csv",
     "Addresses_temporal_concern": "Yes (41,43,47,49)"},
    {"Feature": "TMP5", "Definition": "Min prior receiver-tx count, self-excluded", "Family": "Temporal",
     "Safe_boundary": "<= t (cumsum - 1)", "Source": "TxAddr_edgelist.csv",
     "Addresses_temporal_concern": "Yes (41,43,47,49)"},
    {"Feature": "TMP6", "Definition": "Min time since last activity", "Family": "Temporal",
     "Safe_boundary": "< t (strict, via shift)", "Source": "AddrTx+TxAddr edgelists",
     "Addresses_temporal_concern": "Partial (53,54,55)"},
    {"Feature": "TMP7", "Definition": "Min distinct active time steps", "Family": "Temporal",
     "Safe_boundary": "<= t (cumcount)", "Source": "AddrTx+TxAddr edgelists",
     "Addresses_temporal_concern": "Partial (53,54,55)"},
    {"Feature": "NET1", "Definition": "Historical address-linked tx count, input side", "Family": "Network",
     "Safe_boundary": "<= t, self excluded", "Source": "AddrTx_edgelist.csv",
     "Addresses_temporal_concern": "Indirect (network dim.)"},
    {"Feature": "NET2", "Definition": "Historical address-linked tx count, output side", "Family": "Network",
     "Safe_boundary": "<= t, self excluded", "Source": "TxAddr_edgelist.csv",
     "Addresses_temporal_concern": "Indirect (network dim.)"},
])

dist_stats = final[feature_cols].agg(["count", "min", "max", "mean", "std"]).T
dist_stats["Missing"] = len(final) - dist_stats["count"]
audit_summary = audit_summary.merge(
    dist_stats[["Missing", "min", "max", "mean", "std"]].reset_index().rename(columns={"index": "Feature"}),
    on="Feature", how="left"
)
audit_summary.columns = ["Feature", "Definition", "Family", "Safe_boundary", "Source",
                          "Addresses_temporal_concern", "Missing", "Min", "Max", "Mean", "Std"]
pd.set_option("display.max_colwidth", 60)
display(audit_summary)

Feature                                                                              Definition     Family                          Safe_boundary                  Source Addresses_temporal_concern  Missing      Min         Max      Mean        Std
   FIN3                                           Largest input share = in_BTC_max/in_BTC_total  Financial   None needed -- transaction-intrinsic        txs_features.csv                         No    965.0 0.002716    1.000000  0.916328   0.200416
   FIN4                                        Largest output share = out_BTC_max/out_BTC_total  Financial   None needed -- transaction-intrinsic        txs_features.csv                         No    965.0 0.002008    1.000000  0.881011   0.167313
   FIN5                                                     Fee-to-value ratio = fees/total_BTC  Financial   None needed -- transaction-intrinsic        txs_features.csv                         No    965.0 0.000000   44.825001  0.027090   0.391850
   FIN6 

## Summary

14 transaction-time-safe behavioural features constructed from `txs_features.csv`,
`AddrTx_edgelist.csv`, and `TxAddr_edgelist.csv` only -- no full-history wallet table, no labels,
no future information. All leakage assertions pass, 5 hand-traced transactions (3 deliberately
drawn from known-changed cases) match the pipeline exactly, and the output is a deterministic,
reproducible Parquet file (`transaction_time_safe_features_v1.parquet`).

**Correction applied in this version, per explicit review:** TMP1/TMP2 now use combined any-role
address activity for their age lookup (not role-restricted) -- 3,174 TMP1 values (1.56%) and 688
TMP2 values (0.34%) changed as a result, every single change an increase, consistent with pooling
in additional history. No other feature (FIN3-6, STR1-2, TMP4-7, NET1-2) was touched.

**This does not claim to eliminate all leakage.** It directly addresses the count/age-driven half
of the 13-14 flagged inherited features (TMP1, TMP2, TMP4, TMP5 map onto 53/54/55/41/43/47/49/90);
the value-driven half (23, 25, 29, 30, 65) has no safe replacement here, because per-address BTC
value data does not exist in the available edge files -- documented in Part 1/2 of this project's
design discussion, not silently dropped.

**New finding from this correction:** TMP1/TMP7 correlation rose from 0.955 to **0.999** -- now
effectively perfect, since both are derived from the same combined-activity ranking once TMP1 was
fixed to use it too. Flagged for a future decision on whether to keep both; not resolved
unilaterally here. TMP4/NET1 (0.900) and STR2/TMP4 (0.795) are unchanged by this correction.

**Stopping here, as instructed.** No GNN, no model training, no label-based feature selection
performed in this notebook.